# Sesiones GOFO en Kaggle: Luis apoya a la PC en DFW y DFW02 (reparto y turnos de sesion con la PC);
# Tracking Int revisa SOLO las ciudades indicadas ese dia en la celda 4 (TRACKING_STATIONS).
# 29/10: cada cuenta de Kaggle puede correr una sola sesion (SESION en la celda 1) con la maquina completa.
# Una misma sesion NUNCA en dos cuentas de Kaggle a la vez.
# Requiere subir primero portal/api/session_guard.php y portal/inc/session_guard.php al portal.
# Kaggle: T4 x2, Internet On, dataset PRIVADO con codigo y pod_multisesion_secretos.zip.



In [ ]:
# 1. Codigo y sesiones (conservar storage entre reinicios si se desea reutilizar el cache)
# Sesion de ESTA cuenta de Kaggle: 'luis', 'tracking' o 'luis,tracking' (las dos en la misma maquina).
SESION='luis'
SESIONES=[s.strip() for s in SESION.split(',') if s.strip()]
assert SESIONES and all(s in ('luis','tracking') for s in SESIONES),"SESION debe ser 'luis', 'tracking' o 'luis,tracking'."
WORKERS=['gpu_'+s for s in SESIONES]
import glob,os,zipfile,subprocess,shutil
import base64,io
from pathlib import Path
W=Path('/kaggle/working')
code=glob.glob('/kaggle/input/**/pod_etapa2_codigo.zip',recursive=True)
ROOT=W/'pod_automation'
if code:
    with zipfile.ZipFile(code[0]) as z: z.extractall(W)
else:
    sources=glob.glob('/kaggle/input/**/pod_automation/main.py',recursive=True)
    assert sources,'Falta el codigo actualizado en los datos de entrada.'
    shutil.copytree(Path(sources[0]).parent,ROOT,dirs_exist_ok=True)
# RELEASE_CODE_OVERLAY
assert (ROOT/'tools/run_kaggle_multi.py').exists(),'El dataset contiene codigo anterior; cargar el ZIP multiciudad actualizado.'
import sys
sys.path.insert(0,str(ROOT))
from tools.kaggle_state import restore
resume=glob.glob('/kaggle/input/**/pod_kaggle_estado.zip',recursive=True)
if resume:
    restore(ROOT,resume[0])
    print('Estado anterior restaurado: cache, coordenadas, checkpoints y fotos.',flush=True)
elif not (ROOT/'storage'/'analysis_cache.db').exists():
    print('AVISO: esta maquina no tiene cache previa. Para recuperarla, adjuntar pod_kaggle_estado.zip como Input.',flush=True)
secrets=glob.glob('/kaggle/input/**/pod_multisesion_secretos.zip',recursive=True)
(ROOT/'storage'/'sessions').mkdir(parents=True,exist_ok=True)
if secrets:
    with zipfile.ZipFile(secrets[0]) as z: z.extractall(ROOT/'storage'/'sessions')
for name in SESIONES:
    if not (ROOT/'storage'/'sessions'/(name+'.json')).exists():
        files=glob.glob('/kaggle/input/**/'+name+'.json',recursive=True)
        assert files,'Falta el dataset privado con '+name+'.json'
        shutil.copy(files[0],ROOT/'storage'/'sessions'/(name+'.json'))
print('Sesion(es) de esta cuenta:',', '.join(SESIONES),flush=True)
os.chdir(ROOT)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert len(subprocess.check_output(['nvidia-smi','--query-gpu=index','--format=csv,noheader'],text=True).strip().splitlines())==2,'Se requieren T4 x2.'
print('Codigo listo; storage y coordenadas existentes se conservan.',flush=True)



In [ ]:
# 2. Instalar dependencias en la sesion de Kaggle
import sys,time
def setup_step(label,command,timeout=1200):
    print('INSTALANDO:',label,flush=True)
    started=time.monotonic()
    subprocess.run(command,shell=True,check=True,timeout=timeout)
    print('LISTO:',label,round(time.monotonic()-started,1),'s',flush=True)
setup_step('herramientas del sistema','apt-get -qq update && apt-get -qq install -y tesseract-ocr zstd xvfb')
setup_step('virtualenv y wrapt',f'{sys.executable} -m pip install --progress-bar off virtualenv wrapt')
setup_step('entorno principal','test -d venv || python -m virtualenv venv')
setup_step('dependencias principales','venv/bin/python -m pip install --progress-bar off wrapt -r deploy/ubuntu/requirements-linux.txt')
setup_step('Chromium','venv/bin/python -m playwright install --with-deps chromium')
setup_step('entorno Paddle','test -d venv_paddle || python -m virtualenv venv_paddle')
setup_step('wrapt para Paddle','venv_paddle/bin/python -m pip install --progress-bar off wrapt')
setup_step('Paddle GPU','venv_paddle/bin/python -m pip install --progress-bar off paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/')
setup_step('Paddle OCR','venv_paddle/bin/python -m pip install --progress-bar off -r deploy/ubuntu/requirements-paddle-linux.txt')
setup_step('Ollama','command -v ollama >/dev/null || (curl -fsSL https://ollama.com/install.sh -o /tmp/install-pod-ollama.sh && sh /tmp/install-pod-ollama.sh)')



In [ ]:
# 3. Servicios compartidos: qwen2.5vl:3b en ambas GPU (sin 7b); un Paddle en cada GPU
import time,json,urllib.request
# 29/10: consultas de vision simultaneas por GPU. Un solo valor para Ollama y para la cola del script (GPU_SLOTS_PER_URL),
# asi no pueden quedar distintos. Benchmark 28/10 (4050): 2 = ~255-267 img/min contra ~160-180 con 1, +46 MiB de VRAM.
# Si Ollama da HTTP 500 o falta memoria en la T4, volver a '1' y reiniciar la sesion.
GPU_PARALLEL='2'
os.environ['GPU_SLOTS_PER_URL']=GPU_PARALLEL
services=[]
for g in (0,1):
    envg=dict(os.environ,CUDA_VISIBLE_DEVICES=str(g),OLLAMA_HOST=f'127.0.0.1:{11434+g}',
              OLLAMA_NUM_PARALLEL=GPU_PARALLEL,OLLAMA_MAX_LOADED_MODELS='1',OLLAMA_CONTEXT_LENGTH='2048',
              OLLAMA_FLASH_ATTENTION='1',OLLAMA_KEEP_ALIVE='-1',OMP_NUM_THREADS='1',
              OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',NUMEXPR_NUM_THREADS='1')
    log=(W/f'ollama{g}_multi.log').open('a')
    services.append(subprocess.Popen(['ollama','serve'],env=envg,stdout=log,stderr=subprocess.STDOUT))
    log.close()
    time.sleep(5)
    assert services[-1].poll() is None,f'Ollama GPU {g} no arranco; revisar puerto/log y usar una sesion nueva.'
    for model in ['qwen2.5vl:3b']:   # 28/10: sin el 7b de respaldo; el 3b en las dos GPU
        # capture_output: la barra de progreso de la descarga generaba miles de lineas y dejaba el visor de logs de Kaggle muy retrasado
        subprocess.run(['ollama','pull',model],env=envg,check=True,capture_output=True)
        print(f'Modelo {model} listo en GPU {g}',flush=True)
        data=json.dumps({'model':model,'prompt':'hi','stream':False,'keep_alive':-1,
                         'options':{'num_predict':3,'num_ctx':2048}}).encode()
        with urllib.request.urlopen(urllib.request.Request(f'http://127.0.0.1:{11434+g}/api/generate',data=data),timeout=300) as r:
            r.read()
    envp=dict(os.environ,CUDA_VISIBLE_DEVICES=str(g),OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',NUMEXPR_NUM_THREADS='1')
    log=(W/f'paddle{g}_multi.log').open('a')
    services.append(subprocess.Popen(['venv_paddle/bin/python','tools/paddle_ocr_server.py','--port',str(8770+g)],env=envp,stdout=log,stderr=subprocess.STDOUT))
    log.close()
for g in (0,1):
    for attempt in range(90):
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{8770+g}/health',timeout=5) as r:
                assert r.status==200
            break
        except Exception:
            if attempt==89: raise RuntimeError(f'Paddle GPU {g} no esta listo; revisar su log.')
            time.sleep(2)
print(f'Servicios listos. La cola permite {GPU_PARALLEL} peticion(es) de vision por GPU y cuatro analisis en total.')
subprocess.run(['nvidia-smi','--query-gpu=index,memory.used,memory.total','--format=csv'],check=False)



In [ ]:
# 4. Configurar clave y comprobar archivos; la autenticacion GOFO se comprueba en cada proceso
from kaggle_secrets import UserSecretsClient
key=UserSecretsClient().get_secret('PORTAL_API_KEY')
assert key,'Falta el secreto PORTAL_API_KEY.'
env=dict(os.environ,PORTAL_URL='https://trackingint.com/portal',PORTAL_API_KEY=key,OPERATION_TIMEZONE='America/Caracas')
env['MULTI_STOP_AT']='10:00'     # 28/10 corrida nocturna: cierra a la PROXIMA 10:00 de Venezuela (antes 22:30 de hoy)
env['MULTI_MAX_HOURS']='11'      # tope desde el arranque: Kaggle corta cada sesion a las 12 h y hay que dejar tiempo para guardar el estado
# 29/10 (pedido del usuario): ciudades de Tracking PARA HOY; cambiarlas cada dia. Validas: DFW-BAA, DFW02-BAA, IAH-BAA,
# IAH02-BAA, ORD-BAA. Si coinciden con las de Luis/PC, el reparto del portal evita repetir albaranes.
env['TRACKING_STATIONS']='DFW-BAA,DFW02-BAA'
env['MULTI_SESSIONS']=SESION
args=['venv/bin/python','tools/run_kaggle_multi.py']
for name in SESIONES: args+=['--'+name,f'storage/sessions/{name}.json']
subprocess.run(args+['--check'],env=env,check=True)



In [ ]:
# 5. Lanzar la(s) sesion(es) de esta cuenta: Luis en DFW y DFW02 (con la PC), Tracking en las ciudades de la celda 4
import uuid
env['MULTI_RUN_ID']=uuid.uuid4().hex
assert 'multi_run' not in globals() or multi_run.poll() is not None,'Ya hay una corrida lanzada desde este notebook.'
supervisor_log=(W/'multiciudad.log').open('a')
multi_run=subprocess.Popen(['xvfb-run','-a']+args,env=env,stdout=supervisor_log,stderr=subprocess.STDOUT)
supervisor_log.close()
time.sleep(5)
assert multi_run.poll() is None,'El lanzador se detuvo: revisar /kaggle/working/multiciudad.log (endpoint/cookies).'
print('Corrida iniciada:',', '.join(WORKERS),'. No iniciar otra corrida de Kaggle con estas mismas cookies.')



In [ ]:
# 6. Vigilar y recuperar caidas hasta las 10:00 de Venezuela (o 11 h); requiere que la sesion de Kaggle siga viva
from datetime import datetime
from zoneinfo import ZoneInfo
import signal
from tools.run_kaggle_multi import stop_timestamp
now=datetime.now(ZoneInfo('America/Caracas'))
end=stop_timestamp(now,env)
print('Cierre programado:',datetime.fromtimestamp(end,ZoneInfo('America/Caracas')).strftime('%Y-%m-%d %H:%M'),'(Venezuela)',flush=True)
supervisor_failures=0
# 28/10: diagnostico descargable. Una muestra por minuto (GPU, CPU/RAM, ritmo por sesion) y, cada 20 min y al final, el ZIP
# Output > diagnostico_kaggle.zip (logs de cada sesion, resultados, resumen). Sin cookies, claves ni fotos.
from tools.kaggle_diagnostico import sample,append_sample,build_bundle
from tools.cuello_kaggle import informe_cuello
DIAG=W/'diagnostico'
last_bundle=time.time()
last_cuello=time.time()   # 29/10: cada 10 min, que recurso limita (GPU, CPU, analisis o falta de trabajo)
try:
    while time.time()<end:
        if multi_run.poll() is not None:
            if multi_run.returncode==2:
                print('Las sesiones requieren renovar cookies; no se relanza con credenciales vencidas.',flush=True)
                break
            supervisor_failures+=1
            delay=min(300,30*2**min(supervisor_failures-1,4))
            print(f'Supervisor detenido; relanzando en {delay}s mientras Kaggle siga activo.',flush=True)
            time.sleep(min(delay,max(0,end-time.time())))
            if time.time()>=end: break
            supervisor_log=(W/'multiciudad.log').open('a')
            multi_run=subprocess.Popen(['xvfb-run','-a']+args,env=env,stdout=supervisor_log,stderr=subprocess.STDOUT)
            supervisor_log.close()
            continue
        time.sleep(min(60,max(0,end-time.time())))
        try:
            append_sample(DIAG/'metricas.jsonl',sample(ROOT))
            if time.time()-last_bundle>=1200:
                build_bundle(ROOT,W,W/'diagnostico_kaggle.zip'); last_bundle=time.time()
            if time.time()-last_cuello>=600:
                last_cuello=time.time()
                print(informe_cuello(DIAG/'metricas.jsonl',ROOT,10),flush=True)
        except Exception as error: print('Diagnostico:',type(error).__name__,str(error)[:150],flush=True)
        for worker in WORKERS:
            report_env=dict(env,STORAGE_DIR=str(ROOT/'storage'/'multi'/'workers'/worker))
            report=subprocess.run(['venv/bin/python','tools/activity_report.py','--json'],env=report_env,capture_output=True,text=True,timeout=30)
            print(worker,report.stdout.strip() or report.stderr[-200:],flush=True)
        for worker in WORKERS:
            logpath=ROOT/'storage'/'multi'/'workers'/worker/'logs'/('run_'+datetime.now(ZoneInfo('America/Caracas')).strftime('%Y%m%d')+'.log')
            if logpath.exists():
                with logpath.open('rb') as tail:
                    tail.seek(max(0,logpath.stat().st_size-6000))
                    print(worker,'ULTIMOS EVENTOS:',tail.read().decode('utf-8',errors='replace').splitlines()[-5:],flush=True)
        print(subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout,flush=True)
        cpu=subprocess.run(['venv/bin/python','-c',
            "import psutil,json; print(json.dumps({'cpu_logicas':psutil.cpu_count(),'cpu_porcentaje':psutil.cpu_percent(interval=0.2),'ram_porcentaje':psutil.virtual_memory().percent}))"],
            capture_output=True,text=True,timeout=5)
        print(cpu.stdout.strip(),flush=True)
        for g in (0,1):
            models=subprocess.run(['ollama','ps'],env=dict(env,OLLAMA_HOST=f'127.0.0.1:{11434+g}'),capture_output=True,text=True,timeout=10)
            print(f'Modelos GPU {g}:',models.stdout.strip(),flush=True)
finally:
    state=ROOT/'storage'/'multi'/'supervisor.json'
    if state.exists():
        status=json.loads(state.read_text())
        if status.get('run_id')==env['MULTI_RUN_ID']:
            try: os.kill(status['pid'],signal.SIGTERM)
            except ProcessLookupError: pass
    try: multi_run.wait(timeout=115)
    except subprocess.TimeoutExpired: multi_run.terminate()
    try: print('Diagnostico final (descargar en Output):',build_bundle(ROOT,W,W/'diagnostico_kaggle.zip'),flush=True)
    except Exception as error: print('No se pudo crear el diagnostico:',type(error).__name__,str(error)[:150],flush=True)
    from tools.kaggle_state import backup
    try: print('Guardando estado para la proxima corrida:',backup(ROOT,W/'pod_kaggle_estado.zip'),flush=True)
    except Exception as error: print('No se pudo exportar el estado:',type(error).__name__,str(error),flush=True)
    for service in services:
        if service.poll() is None: service.terminate()
    print('Corridas y servicios detenidos. Para liberar la cuota de Kaggle, detener tambien la sesion del notebook.')
